In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision import models
import torch.nn as nn


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [6]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])



In [7]:
test_dataset = datasets.ImageFolder(
    root="dataset/test",
    transform=transform
)


In [8]:
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
    
)


In [9]:
model = models.resnet18(pretrained=False)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)


C:\Users\Shubham\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Shubham\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [10]:
model.load_state_dict(torch.load("ecg_model.pth"))


<All keys matched successfully>

In [11]:
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [12]:
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()


In [18]:
print("Test Accuracy:", 100 * correct / total)


Test Accuracy: 100.0


In [20]:
from PIL import Image
import torch

# Make sure model is in evaluation mode
model.eval()

def predict_ecg(image_path):

    # Load image
    img = Image.open(image_path).convert('L')

    # Apply same transform used in training
    img = transform(img)

    # Add batch dimension
    img = img.unsqueeze(0)

    img = img.to(device)

    # Prediction
    with torch.no_grad():
        output = model(img)
        _, predicted = torch.max(output, 1)

    classes = ['AbnormalHB', 'Normal']

    print("Prediction:", classes[predicted.item()])


In [25]:
predict_ecg("dataset/test/Normal/norm.png")


Prediction: AbnormalHB
